In [ ]:
import pandas as pd
import numpy as np
import iqplot
from plot_tools import *
import itertools as it
import bokeh.plotting
import bokeh.io
import holoviews as hv
from holoviews import dim, opts
import bokeh.models
from plot_tools import *
from bokeh.layouts import gridplot
from scipy.spatial.distance import jensenshannon
hv.extension('bokeh')


In [ ]:
df_abun = pd.read_csv('e003_round4_assembly_abundances.csv')
df_abun['AC']=df_abun['subject'].transform(lambda x: 'AC' in x)
df_abun=df_abun.loc[~df_abun['AC'],:]
df_abun_get_alpha = df_abun.loc[df_abun['relative_abundance']>=1e-3,:]#.drop(columns='Unnamed:0')
df_abun_get_alpha['counts']=1
df_abun_get_alpha = df_abun_get_alpha.groupby(['sample','mesocosm','passage','type_mesocosm','comm', ]).sum(numeric_only=True).reset_index()
df_abun_get_alpha['subjects'] = df_abun_get_alpha['type_mesocosm'].transform(lambda x: '-'.join(x.split('-')[:2]))
df_abun_get_alpha['env'] = df_abun_get_alpha['type_mesocosm'].transform(lambda x: '-'.join(x.split('-')[2:]))
bad_samples = df_abun_get_alpha.loc[df_abun_get_alpha['counts']<20,:]
bad_samples

df_abun = df_abun.loc[~df_abun['sample'].isin(bad_samples),:]
df_abun = df_abun.loc[df_abun['sample']!='Assembly-G12-fecal-AA-fecal-0_S703',:] # also bad sampleno counts
df_abun_get_alpha = df_abun_get_alpha.loc[~df_abun_get_alpha['sample'].isin(bad_samples),:]
df_abun_get_alpha = df_abun_get_alpha.loc[df_abun_get_alpha['sample']!='Assembly-G12-fecal-AA-fecal-0_S703',:]
df_abun.head()
#df_abun.loc[df_abun['relative_abundance']<1e-3,'relative_abundance']=1e-4
#df_abun_p5 = df_abun.loc[df_abun['passage'].isin([0,5])
df_abun['order']= df_abun['Lineage'].transform(lambda x: x.split(';')[3])
df_abun
df_abun.loc[df_abun['family']=='f__','family']= df_abun.loc[df_abun['family']=='f__','order'] + ' ' + df_abun.loc[df_abun['family']=='f__','family']
#df_abun = df_abun.loc[df_abun['family']!='family',:]
#df_abun['family'].unique()
df_abun_assembly =df_abun_get_alpha#.groupby(['sample','family','subject', 'media','type_mesocosm', 'mesocosm','passage','exp_type','comm']).sum(numeric_only=True).reset_index()
df_abun_assembly.head()
df_abun_assembly_good = df_abun_assembly.loc[(df_abun_assembly['passage']==5),:]
df_abun_assembly_good['comm'].unique()
df_abun_assembly_good = df_abun_assembly_good.loc[df_abun_assembly_good['comm'].isin(['B10','B11','B8','B2','B4','B5']),:]
df_abun_assembly_good.head()

In [ ]:
df_abun = pd.read_csv('e003_coalescence_metadata_round4_abundances.csv')

e003_metadata_good = pd.read_csv('e003_coalescence_metadata_round4_good.csv')
df_abun=df_abun.loc[df_abun['sample'].isin(e003_metadata_good['sample'].values),:]
df_abun['AC']=df_abun['parent_subjects'].transform(lambda x: 'AC' in x)
df_abun=df_abun.loc[~df_abun['AC'],:]
df_abun['relative_abundance_adj'] = 1 - np.exp(-1e3*df_abun['relative_abundance'])
df_abun_get_alpha = df_abun.loc[df_abun['relative_abundance']>=1e-3,:]

#.drop(columns='Unnamed:0')
df_abun_get_alpha['counts']=1
df_abun_get_alpha = df_abun_get_alpha.groupby(['sample','mesocosm','passage','inoculumn_sample','type_mesocosm','parent_subjects']).sum(numeric_only=True).reset_index()
df_abun_get_alpha['subjects'] = df_abun_get_alpha['type_mesocosm'].transform(lambda x: '-'.join(x.split('-')[:2]))
df_abun_get_alpha['env'] = df_abun_get_alpha['type_mesocosm'].transform(lambda x: '-'.join(x.split('-')[2:]))
bad_samples = df_abun_get_alpha.loc[df_abun_get_alpha['counts']<20,:]
df_abun_get_alpha=df_abun_get_alpha.loc[df_abun_get_alpha['counts']>20,:]
df_abun_alpha_good = df_abun_get_alpha.loc[df_abun_get_alpha['type_mesocosm'].isin(['AA-AF-mGAM-mGAM',]),:]


p = bokeh.plotting.figure(width=500,height=200)

in_sample = df_abun_get_alpha.loc[df_abun_get_alpha ['type_mesocosm']=='AA-AA-mGAM-mGAM','inoculumn_sample'].unique()[0]
df_meso =  df_abun_get_alpha.loc[df_abun_get_alpha['sample']==in_sample,:]
p.ray(x=0,y=df_meso['counts'], color=bokeh.palettes.Bright[7][-1],angle=0,alpha=.5,width=2)
p.scatter(0,df_meso['counts'], color=bokeh.palettes.Bright[7][-1],alpha=1,size=10,legend_label='A')

in_sample = df_abun_get_alpha.loc[df_abun_get_alpha ['type_mesocosm']=='AF-AF-mGAM-mGAM','inoculumn_sample'].unique()[0]
df_meso =  df_abun_get_alpha.loc[df_abun_get_alpha['sample']==in_sample,:]
p.ray(x=0,y=df_meso['counts'], color=bokeh.palettes.Light[9][-1],angle=0,alpha=.5,width=2)
p.scatter(0,df_meso['counts'],color=bokeh.palettes.Light[9][-1],alpha=1,size=10,legend_label='C')

for meso in df_abun_alpha_good['mesocosm'].unique()[::-1]:
    in_sample = df_abun_alpha_good .loc[df_abun_alpha_good ['mesocosm']==meso,'inoculumn_sample'].unique()[0]
    df_meso =  df_abun_alpha_good.loc[(df_abun_alpha_good['mesocosm']==meso)+
                (df_abun_alpha_good['sample']==in_sample),:].sort_values(by='passage')
    
    if 'AA-AF' in meso:
        color = bokeh.palettes.Light[4][-1]
        label='AA-AF'
        alpha =1
    elif 'AA-AA' in meso:
        color = bokeh.palettes.Light[9][-1]
        label='AA-AA'
        alpha = .5
        df_meso = df_meso.loc[df_meso['passage']==0,:]
        p.ray(x=0,y=df_meso['counts'], color=color,angle=0,alpha=.5,width=2)
    else:
        color = bokeh.palettes.Bright[7][-1]
        label='AF-AF'
        alpha=.5
        df_meso = df_meso.loc[df_meso['passage']==0,:]
        p.ray(x=0,y=df_meso['counts'], color=color,angle=0,alpha=.5,width=2)
    p.scatter(df_meso['passage'],df_meso['counts'],color = color,size=10, alpha=alpha,legend_label='A-C',)
    p.line(df_meso['passage'],df_meso['counts'],color = color,alpha=alpha)
#
p.xaxis.axis_label_text_font_size='22px'
p.xaxis.major_label_text_font_size='15px'
p.yaxis.axis_label_text_font_size='22px'
p.yaxis.major_label_text_font_size='15px'
p.xaxis.minor_tick_line_color= None
p.xgrid.grid_line_color = None
p.ygrid.grid_line_color = None
p.yaxis.minor_tick_line_color= None
p.y_range = bokeh.models.Range1d(20,70)
p.xaxis.axis_label='Timepoint'
p.xaxis.axis_label_text_font_style='normal'
#p.legend.visible = False

p.output_backend='svg'
export_plot_pdf(p,'num_species')
bokeh.io.show(p)
#p.yaxis.minor_tick_line_color= None

In [ ]:

df_abun_alpha_good = df_abun_get_alpha.loc[df_abun_get_alpha['type_mesocosm'].isin(['AA-AF-mGAM-mGAM',
                                                                                   'AF-AF-mGAM-mGAM',
                                                                                    'AA-AA-mGAM-mGAM',
                                                                                   ]),:]


p = bokeh.plotting.figure(width=500,height=200)
in_sample = df_abun_get_alpha.loc[df_abun_get_alpha ['type_mesocosm']=='AA-AA-mGAM-mGAM','inoculumn_sample'].unique()[0]
df_meso =  df_abun_get_alpha.loc[df_abun_get_alpha['sample']==in_sample,:]
#p.ray(x=0,y=df_meso['counts'], color=bokeh.palettes.Bright[7][-1],angle=0,alpha=.5,width=2)
#p.scatter(0,df_meso['counts'], color=bokeh.palettes.Bright[7][-1],alpha=1,size=10,legend_label='A')

in_sample = df_abun_get_alpha.loc[df_abun_get_alpha ['type_mesocosm']=='AF-AF-mGAM-mGAM','inoculumn_sample'].unique()[0]
df_meso =  df_abun_get_alpha.loc[df_abun_get_alpha['sample']==in_sample,:]
#p.ray(x=0,y=df_meso['counts'], color=bokeh.palettes.Light[9][-1],angle=0,alpha=.5,width=2)
#p.scatter(0,df_meso['counts'],color=bokeh.palettes.Light[9][-1],alpha=1,size=10,legend_label='C')
metric = 'relative_abundance_adj'
for meso in df_abun_alpha_good['mesocosm'].unique()[::-1]:
    in_sample = df_abun_alpha_good .loc[df_abun_alpha_good ['mesocosm']==meso,'inoculumn_sample'].unique()[0]
    df_meso =  df_abun_alpha_good.loc[(df_abun_alpha_good['mesocosm']==meso)+
                (df_abun_alpha_good['sample']==in_sample),:].sort_values(by='passage')
    adj_factor = 0
    if 'AA-AF' in meso:
        color = bokeh.palettes.Light[4][-1]
        label='AA-AF'
        alpha =1
    elif 'AA-AA' in meso:
        color=bokeh.palettes.Bright[7][-1]
        label='AA-AA'
        alpha = .5
        adj_factor = .05
        #df_meso = df_meso.loc[df_meso['passage']==0,:]
      #  p.ray(x=0,y=df_meso['counts'], color=color,angle=0,alpha=.5,width=2)
    else:
        color=bokeh.palettes.Light[9][-1]
        label='AF-AF'
        alpha=.5
        adj_factor = .1
      #  df_meso = df_meso.loc[df_meso['passage']==0,:]
        #p.ray(x=0,y=df_meso['counts'], color=color,angle=0,alpha=.5,width=2)
    p.scatter(df_meso['passage']+adj_factor,df_meso[metric],color = color,size=10, alpha=alpha,legend_label='A-C',)
    if 'AA-AF' in meso:
        p.line(df_meso['passage'],df_meso[metric],color = color,alpha=alpha)
#
p.xaxis.axis_label_text_font_size='22px'
p.xaxis.major_label_text_font_size='15px'
p.yaxis.axis_label_text_font_size='22px'
p.yaxis.major_label_text_font_size='15px'
p.xaxis.minor_tick_line_color= None
p.xgrid.grid_line_color = None
p.ygrid.grid_line_color = None
p.yaxis.minor_tick_line_color= None
#p.y_range = bokeh.models.Range1d(20,70)
p.xaxis.axis_label='Timepoint'
p.xaxis.axis_label_text_font_style='normal'
#p.legend.visible = False

p.output_backend='svg'
p.legend.visible=False
export_plot_pdf(p,'num_species')
bokeh.io.show(p)
#p.yaxis.minor_tick_line_color= None

In [ ]:


metric = 'counts'
df_abun_get_alpha['single_subject']=False
df_abun_get_alpha.loc[df_abun_get_alpha['parent_subjects'].isin(['AA-AA','AE-AE','AF-AF']),'single_subject']=True

df_abun_get_alpha['passageplot']=df_abun_get_alpha['passage'].copy()
df_abun_get_alpha.loc[df_abun_get_alpha['single_subject'],'passageplot']=df_abun_get_alpha.loc[df_abun_get_alpha['single_subject'],'passage'].astype(float)+.2

#p.yaxis.minor_tick_line_color= None

In [ ]:
qs = df_abun_get_alpha.groupby(["single_subject",'passageplot']).counts.quantile([0.25, 0.5, 0.75])


In [ ]:
p = hv.Points(df_abun_get_alpha, vdims = [metric, 'single_subject'],
              kdims = ['passageplot',metric]).opts(color = 'single_subject',
                                                                 cmap = [bokeh.palettes.Light[4][-1],
                                                                         bokeh.palettes.Light[9][-1]],
                                                                 jitter=.1,size = 5,alpha = .5,
                                                                width=300,height=150,ylim=(0,100)
                                                 )

p = hv.render(p)
#cmap = factor_cmap("kind", "TolRainbow7", kinds)
# compute quantiles
qs = df_abun_get_alpha.groupby(["single_subject",'passageplot']).counts.quantile([0.25, 0.5, 0.75])
qs = qs.unstack().reset_index()
qs.columns = ["single_subject",'passageplot', "q1", "q2", "q3"]
df = pd.merge(df_abun_get_alpha, qs, on=["single_subject",'passageplot'], how="left")
source = bokeh.models.ColumnDataSource(df)
p.vbar("passageplot", 0.2, "q2", "q3", source=source, color=None, line_color="grey",line_alpha=.5)
p.vbar("passageplot", 0.2, "q1", "q2", source=source, color=None, line_color="grey",line_alpha=.5)
p.legend.visible=False

p.xaxis.axis_label_text_font_size='22px'
p.xaxis.major_label_text_font_size='15px'
p.yaxis.axis_label_text_font_size='22px'
p.yaxis.major_label_text_font_size='15px'
p.xaxis.minor_tick_line_color= None
p.xgrid.grid_line_color = None
p.ygrid.grid_line_color = None
p.yaxis.minor_tick_line_color= None
p.yaxis.axis_label='# Sp'
#p.y_range = bokeh.models.Range1d(20,70)
p.xaxis.axis_label='Timepoint'
p.xaxis.axis_label_text_font_style='normal'
p.yaxis.axis_label_text_font_style='normal'
p.legend.visible = False

p.output_backend='svg'
#p.legend.visible=False
export_plot_pdf(p,'num_species')
bokeh.io.show(p)
#bokeh.io.show(p)

## now trying to make this violin plot

In [ ]:
from numpy.random import choice,shuffle,normal,uniform,binomial
from scipy.stats import gaussian_kde, linregress

In [ ]:
def make_violin_points(ys, X, width, color):
    kernel = gaussian_kde(ys)
    theory_ys = np.linspace(ys.min(),ys.max(),100)
    theory_pdf = kernel(theory_ys)
    #[0.3,0.3,0.1,0.3]
    width = .1
    scale = width/theory_pdf.max()
    
    
    xs = uniform(-1,1,size=len(ys))*kernel(ys)*scale
    
    q25 = np.quantile(ys,0.25)
    q50 = np.quantile(ys,0.5)
    q75 = np.quantile(ys,0.75)
    other_width = width+0.1
    
  #  X = 7.
    
    a1 = hv.Area((theory_ys,X-theory_pdf*scale, X+theory_pdf*scale, ), vdims=['y', 'y2']).opts(invert_axes=True,xlim=(0,100),
                                                                                              fill_color = color, fill_alpha = .2,
                                                                                              width=500, height=250,line_color=None,
                                                                                              ylim = (-1.25,7.5),)
    p1 = hv.Points((ys,X+xs)).opts(color=color,size=5, width=500, height=250,)
    return a1,p1


In [ ]:
df_abun_get_alpha.loc[((df_abun_get_alpha['passageplot']==7.2)*(df_abun_get_alpha['single_subject'])),
'counts'].values

In [ ]:
ps = []
adj = .1
for passage in df_abun_get_alpha['passage'].unique():
    
    list1 = df_abun_get_alpha.loc[(df_abun_get_alpha['passage']==passage)*(df_abun_get_alpha['single_subject']),
'counts'].values
    ys = list1
    X = passage + adj
    width = .1
    color = bokeh.palettes.Light[9][-1]
    a1,p1 = make_violin_points(ys, X, width, color)

    list1 = df_abun_get_alpha.loc[(df_abun_get_alpha['passage']==passage)*(~df_abun_get_alpha['single_subject']),
'counts'].values
    ys = list1
    X = passage - adj
    width = .1
    color = bokeh.palettes.Light[4][-1]
    a2,p2 = make_violin_points(ys, X, width, color)
    p3 = a1*p1*a2*p2
    ps.append(p3)


list1 = df_abun_assembly_good['counts'].values
ys = list1
X = -1
width = .1 
color = bokeh.palettes.Light[9][-1]
a1,p1 = make_violin_points(ys, X, width, color)
ps.append(a1*p1)

In [ ]:
tick_font_size ='22px'
label_font_size = '28px'
p = ps[8]*ps[0]*ps[1]*ps[2]*ps[0]*ps[1]*ps[3]*ps[4]*ps[5]*ps[6]*ps[7]
p.opts(xticks=7)
p = hv.render(p)
df_abun_get_alpha.loc[df_abun_get_alpha['single_subject'],'passageplot']=\
    df_abun_get_alpha.loc[df_abun_get_alpha['single_subject'],'passage'] + adj
df_abun_get_alpha.loc[~df_abun_get_alpha['single_subject'],'passageplot']=\
    df_abun_get_alpha.loc[~df_abun_get_alpha['single_subject'],'passage'] - adj
qs = df_abun_get_alpha.groupby(["single_subject",'passageplot']).counts.quantile([0.25, 0.5, 0.75])
qs = qs.unstack().reset_index()
qs.columns = ["single_subject",'passageplot', "q1", "q2", "q3"]
df = pd.merge(df_abun_get_alpha, qs, on=["single_subject",'passageplot'], how="left")
source = bokeh.models.ColumnDataSource(df)
p.vbar("passageplot", 0.2, "q2", "q3", source=source, color=None, line_color="grey",line_alpha=.5)
p.vbar("passageplot", 0.2, "q1", "q2", source=source, color=None, line_color="grey",line_alpha=.5)

df_abun_assembly_good['passageplot']=-1
qs = df_abun_assembly_good.groupby(['passageplot']).counts.quantile([0.25, 0.5, 0.75])
qs = qs.unstack().reset_index()
qs.columns = ['passageplot', "q1", "q2", "q3"]
df = pd.merge(df_abun_assembly_good, qs, on=['passageplot'], how="left")
source = bokeh.models.ColumnDataSource(df)
p.vbar("passageplot", 0.2, "q2", "q3", source=source, color=None, line_color="grey",line_alpha=.5)
p.vbar("passageplot", 0.2, "q1", "q2", source=source, color=None, line_color="grey",line_alpha=.5)


p.legend.visible=False
p.xaxis.axis_label_text_font_size=label_font_size
p.xaxis.major_label_text_font_size=tick_font_size
p.yaxis.axis_label_text_font_size=label_font_size
p.yaxis.major_label_text_font_size=tick_font_size
p.xaxis.minor_tick_line_color= None
p.yaxis.minor_tick_line_color= None
p.output_backend='svg'
export_plot_pdf(p,'sp_over_time_violin_points_num_sp')
bokeh.io.show(p)

In [ ]:
p = hv.Points(df_abun_get_alpha, vdims = [metric, 'single_subject'],
              kdims = ['passageplot',metric]).opts(color = 'single_subject',
                                                                 cmap = [bokeh.palettes.Light[4][-1],
                                                                         bokeh.palettes.Light[9][-1]],
                                                                 jitter=.1,size = 5,alpha = .5,
                                                                width=300,height=150,ylim=(0,100)
                                                 )

p = hv.render(p)
#cmap = factor_cmap("kind", "TolRainbow7", kinds)
# compute quantiles
qs = df_abun_get_alpha.groupby(["single_subject",'passageplot']).counts.quantile([0.25, 0.5, 0.75])
qs = qs.unstack().reset_index()
qs.columns = ["single_subject",'passageplot', "q1", "q2", "q3"]
df = pd.merge(df_abun_get_alpha, qs, on=["single_subject",'passageplot'], how="left")
source = bokeh.models.ColumnDataSource(df)
p.vbar("passageplot", 0.2, "q2", "q3", source=source, color=None, line_color="grey",line_alpha=.5)
p.vbar("passageplot", 0.2, "q1", "q2", source=source, color=None, line_color="grey",line_alpha=.5)
p.legend.visible=False

p.xaxis.axis_label_text_font_size='22px'
p.xaxis.major_label_text_font_size='15px'
p.yaxis.axis_label_text_font_size='22px'
p.yaxis.major_label_text_font_size='15px'
p.xaxis.minor_tick_line_color= None
p.xgrid.grid_line_color = None
p.ygrid.grid_line_color = None
p.yaxis.minor_tick_line_color= None
p.yaxis.axis_label='# Sp'
#p.y_range = bokeh.models.Range1d(20,70)
p.xaxis.axis_label='Timepoint'
p.xaxis.axis_label_text_font_style='normal'
p.yaxis.axis_label_text_font_style='normal'
p.legend.visible = False

p.output_backend='svg'
#p.legend.visible=False
export_plot_pdf(p,'num_species')
bokeh.io.show(p)
#bokeh.io.show(p)

In [ ]:
from scipy.stats import permutation_test

def statistic(x, y, axis):
    return np.median(x, axis=axis) - np.median(y, axis=axis)

from scipy.stats import permutation_test
passage = []
pvalues = []
stats = []
for p in df_abun_get_alpha['passage'].unique():
    x = df_abun_get_alpha.loc[(df_abun_get_alpha['passage']==p)*(df_abun_get_alpha['single_subject']),'counts'].values
    y = df_abun_get_alpha.loc[(df_abun_get_alpha['passage']==p)*(~df_abun_get_alpha['single_subject']),'counts'].values
    res = permutation_test((x, y), statistic, vectorized=True,permutation_type='independent',
                       n_resamples=10000, alternative='two-sided')
   # print(p,res)
    passage.append(p)
    pvalues.append(res.pvalue)
df = pd.DataFrame(data = {'passage':passage,'pval':pvalues})
df.sort_values(by='passage')